# Baseline Analysis

Loads the preprocessing checkpoint, runs the analysis pipeline, and explores all three baseline methods:

1. **Sliding Window** — finds the lowest-variance window in each waveform
2. **Fixed Window** — divides waveform into fixed windows, picks lowest-variance
3. **First Window** — always uses the first `WINDOW_LEN` samples

The CFD notebook uses the First Window method as its baseline input.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib ipympl

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.signal import welch
from pathlib import Path

from naludaq.board import Board

from toolbox.filters.analysis_pipeline import AnalysisPipeline
from toolbox.filters.steps import (
    rollover_step, spike_step, signal_noise_step, noise_step, baseline_step
)
from toolbox.plotting.save_plots import save_plot
from toolbox.plotting.interactive_plot import interactive_plot


## Configuration

| Variable | Description |
|---|---|
| `BOARD` | `"trbhm"` or `"asoc"` — must match the preprocessing checkpoint |
| `CHUNK_SIZE` | Events per chunk when running chunked; `None` loads all at once |
| `ROLLOVER_THRESHOLD` | ADC samples below this are treated as rollover |
| `SIGNAL_THRESHOLD` | ADC value a signal channel must exceed to be classified as signal |
| `TRIG_THRESHOLD` | ADC value the trigger channel must fall below to be classified as signal |
| `CW_NOISE_FREQS` | Notch filter frequencies in Hz — see PSD section to identify these |
| `WINDOW_LEN` | Baseline window length in samples (one acquisition window = 64 samples) |
| `N_EXAMPLE_EVENTS` | Events shown in the waveform example plots |

In [ ]:
BOARD = "trbhm"  

CHUNK_SIZE = None 

ROLLOVER_THRESHOLD = -2000
SIGNAL_THRESHOLD   =  100   # signal channels, rising edge
TRIG_THRESHOLD     = -500   # trigger channel, falling edge

# Notch filter frequencies for CW noise (Hz).
# Run the PSD section first if unsure — peaks in the noise PSD identify these.
CW_NOISE_FREQS = [1.1718750e9, 1.3281250e9, 1.5625000e9]

WINDOW_LEN       = 64 
N_EXAMPLE_EVENTS = 3

# Derived
CHANNELS         = list(range(8))
SIGNAL_CHANNELS  = list(range(7))
TRIGGER_CHANNEL  = 7

board         = Board("trbhm") if BOARD == "trbhm" else Board("aodsoc_asoc")
SAMPLING_RATE = board.params["samplingrate"] * 1e9  # Hz

CHECKPOINT_FOLDER = Path().cwd() / "data"
CHECKPOINT_FILE   = f"{BOARD}-preprocessed"

PLOTS_DIR = Path("plots") / "baseline"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## Load Checkpoint

In [ ]:
def load_checkpoint(checkpoint_folder: Path | str , checkpoint_file: Path | str) -> dict:
    "Load preprocessed data or results as mmaps"
    checkpoint = {}
    for f in sorted(checkpoint_folder.glob(f"{checkpoint_file}-*.npy")):
        key = f.stem.replace(f"{CHECKPOINT_FILE}-", "")
        try:
            checkpoint[key] = np.load(f, mmap_mode="r", allow_pickle=True)
            print(f"Loaded  {key:30s}  {checkpoint[key].shape}")
        except Exception:
            checkpoint[key] = np.load(f, allow_pickle=True)
            print(f"Loaded  {key:30s}  (pickle)")
    return checkpoint

checkpoint = load_checkpoint(CHECKPOINT_FOLDER, CHECKPOINT_FILE)

data = checkpoint["data"]   # (N, C, S) pedestal-corrected
peds = checkpoint["peds"]   # (C, windows, 64)
print(f"\nEvents: {data.shape[0]}  Channels: {data.shape[1]}  Samples: {data.shape[2]}")


## Interactive Plot

In [ ]:
interactive_plot(data)

## Build Analysis Pipeline

Steps in order:

1. **Rollover** — correct ADC underflow by adding 4096
2. **Spikes** — interpolate over digitizer spike artefacts
3. **Signal / Noise** — classify events; needed for PSD on clean noise events
4. **CW Noise Filter** — notch filter at `CW_NOISE_FREQS`, output saved as `noise_filtered`
5. **Baseline × 3** — sliding window, fixed window, first window (all read `noise_filtered`)

In [ ]:
pipe = AnalysisPipeline()

pipe.add(rollover_step,
         min_threshold=ROLLOVER_THRESHOLD, apply_correction=True, correction=4096)

pipe.add(spike_step,
         source_key="rollover_corrected",
         apply_correction=True, 
         baseline_lookback_window=2)

pipe.add(signal_noise_step,
         source_key="spike_filtered",
         signal_threshold=SIGNAL_THRESHOLD, trig_threshold=TRIG_THRESHOLD,
         is_sig_rising_edge=True, is_trig_rising_edge=False, in_place=True)

pipe.add(noise_step,
         source_key="spike_filtered", save_key="noise_filtered",
         f_s=SAMPLING_RATE, filter_type="notch", q_factor=10, freqs=CW_NOISE_FREQS)

pipe.add(baseline_step,
         source_key="noise_filtered", method=0,
         window_len=WINDOW_LEN, sliding_window_step=WINDOW_LEN)

pipe.add(baseline_step,
         source_key="noise_filtered", method=1,
         window_len=WINDOW_LEN)

pipe.add(baseline_step,
         source_key="noise_filtered", method=2,
         window_len=WINDOW_LEN, window_index=0, axis=2)

print(pipe)


## Run Pipeline

In [ ]:
results = pipe.run(data, chunk_size=CHUNK_SIZE)

# Sanity check
print("\nResult keys and shapes:")
for k, v in results.items():
    if isinstance(v, np.ndarray):
        nan_frac = np.isnan(v.astype(float)).mean() if np.issubdtype(v.dtype, np.floating) else 0.0
        print(f"  {k:45s}  {str(v.shape):20s}  NaN%={nan_frac:.1%}")
    else:
        print(f"  {k:45s}  {type(v).__name__}")


In [ ]:
interactive_plot(results["noise_filtered"])

---
## CW Noise — PSD

Power spectral density of the noise events (before the notch filter is applied).
Sharp peaks identify the CW interference frequencies to set in `CW_NOISE_FREQS`.
If the peaks match your `CW_NOISE_FREQS` values the notch filter is tuned correctly.

In [ ]:
noise_data = np.asarray(results["noise_data"])   # (M, S)

if noise_data.size == 0:
    print("No noise events found — check SIGNAL_THRESHOLD / TRIG_THRESHOLD")
else:
    freqs_hz, psd = welch(noise_data, SAMPLING_RATE, nperseg=min(256, noise_data.shape[-1]), axis=-1)
    mean_psd = np.nanmean(psd, axis=0)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.semilogy(freqs_hz / 1e9, mean_psd, linewidth=0.8)
    for f in CW_NOISE_FREQS:
        ax.axvline(f / 1e9, color="red", linestyle="--", linewidth=1.2, label=f"{f/1e9:.4f} GHz")
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("PSD")
    ax.set_title(f"{BOARD.upper()} — Mean Noise PSD  (N={noise_data.shape[0]} waveforms)")
    ax.legend(fontsize="small")
    plt.tight_layout()
    save_plot(fig, PLOTS_DIR / f"{BOARD}-noise-psd")
    plt.show()
    print(f"Noise events used for PSD: {noise_data.shape[0]}")


---
## Method 1 — Sliding Window

For each event and channel, slides a window of `WINDOW_LEN` samples across the waveform and picks the window with the lowest variance as the baseline region.

In [ ]:
b_sliding = np.asarray(results["baselines_sliding_window"])          # (C, N)
b_sliding_start = np.asarray(results["baseline_sliding_window_start_idx"])  # (N, C)
noise_filtered = np.asarray(results["noise_filtered"])               # (N, C, S)

# --- Distribution per channel ---
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
for i, ch in enumerate(CHANNELS):
    ax = axes[i // 4, i % 4]
    vals = b_sliding[i]
    finite = vals[np.isfinite(vals)]
    ax.hist(finite, bins=60, color="steelblue" if ch < 7 else "darkorange")
    ax.set_title(f"Ch {ch}" + (" [trig]" if ch == TRIGGER_CHANNEL else ""))
    ax.set_xlabel("Baseline (ADC)")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%g"))
fig.suptitle(f"{BOARD.upper()} — Sliding Window Baseline Distribution")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-baseline-sliding-dist")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.title(f"{BOARD.upper()} — Sliding Window Baselines Across Events")
for i, ch in enumerate(CHANNELS):
    vals = b_sliding[i]
    finite = vals[np.isfinite(vals)]
    plt.scatter(np.arange(len(finite)), finite, label=f"Channel {ch}")
    plt.xlabel("Event Index")
    plt.ylabel("ADC Counts")
plt.legend(loc="upper right")
plt.show()

## Method 2 — Fixed Window

Divides the waveform into non-overlapping windows of `WINDOW_LEN` samples and picks the window with the lowest variance.

In [ ]:
b_fixed     = np.asarray(results["baselines_fixed_window"])         # (C, N)
b_fixed_idx = np.asarray(results["baseline_fixed_window_win_idx"])  # (C, N)

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
for i, ch in enumerate(CHANNELS):
    ax = axes[i // 4, i % 4]
    vals = b_fixed[i]
    finite = vals[np.isfinite(vals)]
    ax.hist(finite, bins=60, color="steelblue" if ch < 7 else "darkorange")
    ax.set_title(f"Ch {ch}" + (" [trig]" if ch == TRIGGER_CHANNEL else ""))
    ax.set_xlabel("Baseline (ADC)")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%g"))
fig.suptitle(f"{BOARD.upper()} — Fixed Window Baseline Distribution")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-baseline-fixed-dist")
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.title(f"{BOARD.upper()} — Fixed Window Baselines Across Events")
for i, ch in enumerate(CHANNELS):
    vals = b_fixed[i]
    finite = vals[np.isfinite(vals)]
    plt.scatter(np.arange(len(finite)), finite, label=f"Channel {ch}")
    plt.xlabel("Event Index")
    plt.ylabel("ADC Counts")
plt.legend(loc="upper right")
plt.show()

## Method 3 — First Window

Always uses the first `WINDOW_LEN` samples as the baseline region.  This is the simplest method and the one used as input to the CFD notebook.

In [ ]:
b_first = np.asarray(results["baselines_0_window"])  # (C, N)

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
for i, ch in enumerate(CHANNELS):
    ax = axes[i // 4, i % 4]
    vals = b_first[i]
    finite = vals[np.isfinite(vals)]
    ax.hist(finite, bins=60, color="steelblue" if ch < 7 else "darkorange")
    ax.set_title(f"Ch {ch}" + (" [trig]" if ch == TRIGGER_CHANNEL else ""))
    ax.set_xlabel("Baseline (ADC)")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%g"))
fig.suptitle(f"{BOARD.upper()} — First Window Baseline Distribution")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-baseline-first-dist")
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.title(f"{BOARD.upper()} — Fixed Window Baselines Across Events")
for i, ch in enumerate(CHANNELS):
    vals = b_first[i]
    finite = vals[np.isfinite(vals)]
    plt.scatter(np.arange(len(finite)), finite, label=f"Channel {ch}")
    plt.xlabel("Event Index")
    plt.ylabel("ADC Counts")
plt.legend(loc="upper right")
plt.show()

## Method Comparison

Mean ± std of the baseline value per channel for each method.  Methods that agree closely on noise events suggest a stable baseline.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

COLORS = {                
    "first_window": "#ff7f0e",        
    "sliding_window": "#2ca02c",      
    "windowed_baseline": "#d62728",  
}

fig = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=[f"Channel {i}" for i in range(8)],
    shared_xaxes=True,
)

showlegend = True

for row in range(4):
    for col in range(2):
        idx = row * 2 + col

        fig.add_trace(
            go.Histogram(
                x=b_first[idx],
                nbinsx=32,
                name="First Window",
                legendgroup="first_window",
                showlegend=showlegend,
                marker=dict(color=COLORS["first_window"])
            ),
            row=row + 1,
            col=col + 1,
        )

        fig.add_trace(
            go.Histogram(
                x=b_sliding[idx],
                nbinsx=32,
                name="Sliding Window (Lowest Variance)",
                legendgroup="sliding_window",
                showlegend=showlegend,
                marker=dict(color=COLORS["sliding_window"])
            ),
            row=row + 1,
            col=col + 1,
        )

        fig.add_trace(
            go.Histogram(
                x=b_fixed[idx],
                nbinsx=32,
                name="Windowed Baseline",
                legendgroup="windowed_baseline",
                showlegend=showlegend,
                marker=dict(color=COLORS["windowed_baseline"])
            ),
            row=row + 1,
            col=col + 1,
        )

        if showlegend:
            showlegend = False

fig.update_layout(
    title_text=f"{BOARD.upper()} - Baselines ",
    height=1200,
    width=1300,
    showlegend=True,
)

fig.update_xaxes(title_text="Baseline")
fig.update_yaxes(title_text="Counts", type="log")

save_plot(fig, PLOTS_DIR / f"{BOARD}-baseline-comparison")

fig.show()


In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(12, 10),
    sharex=True,
    sharey=True,
)

fig.suptitle(f"{BOARD.upper()} - Baselines")

baseline_sets = [
    ("Sliding Window", b_sliding),
    ("Fixed Window", b_fixed),
    ("First Window", b_first),
]

for ax, (title, baselines) in zip(axes, baseline_sets):
    for idx in range(8):
        data = baselines[idx]
        valid = ~np.isnan(data)

        x = np.arange(len(data))[valid]
        y = data[valid]

        ax.scatter(x, y, s=8, alpha=0.5)
        ax.plot(x, y, linewidth=0.8, alpha=0.8)

    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Event Index")
axes[1].set_ylabel("Baseline (ADC Counts)")

handles = [
    plt.Line2D([], [], color=plt.cm.tab10(i), label=f"Ch {i}")
    for i in range(8)
]
axes[0].legend(handles=handles, ncol=4, fontsize=9, loc="upper right")

fig.tight_layout()
plt.show()

## Save Results

Saves the baseline results to disk so other notebooks (e.g. CFD) can load them without re-running the pipeline.

In [ ]:
RESULTS_DIR = Path("data") / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_keys = [
    "noise_filtered",
    "baselines_sliding_window",
    "baseline_sliding_window_start_idx",
    "baselines_fixed_window",
    "baseline_fixed_window_win_idx",
    "baselines_0_window",
    "signal_events", "noise_events",
    "signal_channels", "noise_channels",
]

for key in save_keys:
    if key not in results:
        print(f"Skip  {key}  (not in results)")
        continue
    out = RESULTS_DIR / f"{BOARD}-baseline-{key}.npy"
    np.save(out, np.asarray(results[key]))
    print(f"Saved {key:45s}  {out.name}")
